In [1]:
import h5py
import numpy as np
import os
from IPython import display
from pathlib import Path
from corpus.binaural_attention_h5 import BinauralAttentionDataset
import yaml


In [2]:
#### Config path 
cfg_path = "config/binaural_attn/word_task_v10_backbone_word_config.yaml"
config = yaml.safe_load(open(cfg_path, 'r'))
corpus_config = config['corpus']
corpus_config

{'name': 'spatialized_commonvoice_audioset_scenes',
 'cue_type': 'mixed',
 'task': 'word',
 'root': '/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/',
 'mixture_percentages': {'voice_only': 0.5, 'voice_and_location': 0.5},
 'gender_balanced_4M': True,
 'cue_free_percentage': 0.1,
 'v06': True}

In [3]:
n_total = len(BinauralAttentionDataset(**corpus_config, batch_size=1))
dataset = BinauralAttentionDataset(**corpus_config, batch_size=10)

# dummy init
_ = dataset[0]

Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
735 files in train concat dataset
Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
735 files in train concat dataset


In [83]:
from tqdm.auto import tqdm, trange

In [69]:

dataset_counts = np.zeros(len(dataset.all_hdf5_files))

for d_ix in trange(dataset_counts.shape[0]):
    ds = dataset.all_hdf5_datasets[d_ix]
    # dummy init dataset
    # ds[0]
    h5 = ds.dataset
    n_speech_distractors = h5['n_speech_distractors'][:]
    dataset_counts[d_ix] = sum((n_speech_distractors == 0))

100%|██████████| 735/735 [00:01<00:00, 541.25it/s]


In [74]:
dataset_counts.sum() / n_total

np.float64(0.3251319015175238)

In [ ]:
import importlib
import corpus.binaural_word_rec_h5 as word_rec_ds
import torch
importlib.reload(word_rec_ds)
BinauralWordRecDataset = word_rec_ds.BinauralWordRecDataset


In [ ]:
dataset_words = BinauralWordRecDataset(**corpus_config, batch_size=1)
class_map = dataset_words.class_map()
ix_to_word = {v: k for k, v in class_map.items()}

/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
735 files in train concat dataset


In [89]:
# for d_ix in trange(len(dataset_words.all_hdf5_files)):
#     ds = dataset.all_hdf5_datasets[d_ix]
#     # dummy init dataset
#     ds[0]


In [127]:
from typing import List
import src.audio_transforms as at

audio_transforms = at.AudioCompose([
    at.AudioToTensor(),
    at.BinauralCombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'],
                                        high_snr=config['noise_kwargs']['high_snr'],
                                        v2_demean=True),
    at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
])


def collate_fn(samples: List):
    # print(len(samples))
    # samples is a single-element list holding a tuple batches
    scenes = []
    labels = []
    for (_, target, distractor, label) in samples:
        scene_features, _ = audio_transforms(target, None)
        scenes.append(scene_features)
        labels.append(label)
    scene_features = torch.stack(scenes, dim=0)  # stack along batch dimension
    labels = torch.LongTensor(labels)  # stack along batch dimension
    return None, None, scene_features, labels
        
dataloader = torch.utils.data.DataLoader(dataset_words, batch_size=20, num_workers=4, collate_fn=collate_fn)

/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [129]:
from tqdm.auto import tqdm 

for batch in tqdm(dataloader):
    # print(batch[3])
    # break
    continue

  0%|          | 20/59754 [00:05<4:57:40,  3.34it/s]


KeyboardInterrupt: 

In [47]:
from IPython.display import display, Audio

In [106]:
batch[3]

tensor([103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103,
        103, 103, 103, 103, 103, 103])

In [ ]:
class_map


In [135]:

eg_ix = 5
print("Word: ", ix_to_word[batch[3][eg_ix].item()])
Audio(batch[2][eg_ix], rate = 44100)

Word:  cemetery


In [16]:
batch[0][1].shape

(2, 110250)

In [81]:
#Dev get good indices 
safe_ixs = np.where(h5['n_speech_distractors'][:] == 0)[0]
safe_ixs

array([   4,    7,   10, ..., 4991, 4998, 4999])

In [75]:
dataset_counts.sum()

np.float64(1195074.0)

In [34]:
n_speech_distractors = h5['n_speech_distractors'][:]
n_babble = sum((n_speech_distractors >= 6))
n_babble

TypeError: 'NoneType' object is not subscriptable

In [10]:
cue, target, bg, label = dataset[100]

[9]


In [13]:
cue[9], bg[9]

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32))

In [14]:
target[9]

array([[ 2.5545494e-04,  2.0975040e-04,  1.6449876e-04, ...,
         1.6892853e-05, -5.5245124e-04, -1.1010909e-03],
       [ 1.3599879e-04,  2.1810611e-04,  4.0049959e-04, ...,
        -3.5504391e-03, -3.1160549e-03, -2.3812698e-03]], dtype=float32)